# Analysis for the $\sin$ benchmark

The datasets are created as follows.

1. `sin-noise-0`:
    Each $x_j$ is selected uniformly at random between $-\pi$ and $\pi$, $j = 1 \dots 100$.
    Then $y_j = \sin x_j$.
2. `sin-noise-1`:
    Same process as `sin-noise-0` but $y_j = \sin x_j + \eta_j$ where $\eta_j$ has a normal distribution with mean $0$ and standard deviation $0.1$.
3. `sin-noise-2`:
    Same process as `sin-noise-1` but the standard deviation is $0.01$.
4. `sin-wide-noise-k`:
    Same processes as `sin-noise-k` but with $x_j$ drawn uniformly between $-2\pi$ and $2\pi$, $j = 1 \dots 200$.


- TODO Make titles spiffy
- TODO Save to files

## Prelude

In [ ]:
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import sympy

import AnalysisUtils as au

## Loading data

These are the datasets to be fit.

In [ ]:
sn0 = pd.read_csv("datasets/sin-noise-0.csv")
sn1 = pd.read_csv("datasets/sin-noise-1.csv")
sn2 = pd.read_csv("datasets/sin-noise-2.csv")
sw0 = pd.read_csv("datasets/sin-wide-noise-0.csv")
sw1 = pd.read_csv("datasets/sin-wide-noise-1.csv")
sw2 = pd.read_csv("datasets/sin-wide-noise-2.csv")

In [ ]:
fig = sns.relplot(data=sn0, x="x", y="y", color="blue", aspect=1.5)
fig.set(title=r"$y = \sin x$")
au.savefig(fig, file_stem="sin-noise-0")

In [ ]:
fig = sns.relplot(data=sn2, x="x", y="y", color="blue", aspect=1.5)
fig.set(title=r"$y = \sin x + \eta$ where $\eta \sim \mathcal{N}(0, 0.01)$")
au.savefig(fig, file_stem="sin-noise-001")

In [ ]:
fig = sns.relplot(data=sn1, x="x", y="y", color="blue", aspect=1.5)
fig.set(title=r"$y = \sin x + \eta$ where $\eta \sim \mathcal{N}(0, 0.1)$")
au.savefig(fig, file_stem="sin-noise-01")

These are the results of Jessamine symbolic regression.

In [ ]:
full_report = pd.read_csv("Generated/full-report.csv")

In [ ]:
full_report["sympy"] = full_report.expr_original_syms.apply(lambda e: au.parse_if_needed(e))
full_report["complexity"] = full_report.sympy.apply(lambda e: au.complexity(e))
full_report["sympy_defuzz"] = full_report.sympy.apply(lambda e: au.replace_near_integer(e))
full_report["complexity_defuzz"] = full_report.sympy_defuzz.apply(lambda e: au.complexity(e))

In [ ]:
full_report.sort_values(by=["run_set", "data_set", "mse"], inplace=True)

Lop0 = 0.0
Lop4 = 1.0e-4
Lop2 = 1.0e-2
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715", "Lop"] = Lop0
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L4", "Lop"] = Lop4
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L2", "Lop"] = Lop2

# These have to be strings because they need to be exact categorical labels for plotting
spiffy_Lopstr0 = r"$\lambda = 0$"
spiffy_Lopstr4 = r"$\lambda = 10^{-4}$"
spiffy_Lopstr2 = r"$\lambda = 10^{-2}$"
Lopstr0 = "L0000"
Lopstr4 = "L0001"
Lopstr2 = "L0100"
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715", "Lopstr"] = Lopstr0
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L4", "Lopstr"] = Lopstr4
full_report.loc[full_report.run_set=="SRB-2026-06-25-1715-L2", "Lopstr"] = Lopstr2


In [ ]:
full_report.data_set.unique()

In [ ]:
fr2 = full_report.set_index(["data_set", "Lopstr", "sample_num"])
fr2.sort_index(inplace=True)

In [ ]:
fr2

In [ ]:
results_sn0 = fr2.loc["sin-noise-0"]
results_sn1 = fr2.loc["sin-noise-1"]
results_sn2 = fr2.loc["sin-noise-2"]
results_sw0 = fr2.loc["sin-wide-noise-0"]
results_sw1 = fr2.loc["sin-wide-noise-1"]
results_sw2 = fr2.loc["sin-wide-noise-2"]

## For reference

Symbolic regression is fitting noise if it gets MSE any lower than these.

In [ ]:
sn0_mse_bound = au.mse(np.sin(sn0.x), sn0.y)
sn1_mse_bound = au.mse(np.sin(sn1.x), sn1.y)
sn2_mse_bound = au.mse(np.sin(sn2.x), sn2.y)
sw0_mse_bound = au.mse(np.sin(sw0.x), sw0.y)
sw1_mse_bound = au.mse(np.sin(sw1.x), sw1.y)
sw2_mse_bound = au.mse(np.sin(sw2.x), sw2.y)


In [ ]:
mse_bounds = pd.DataFrame({
    "data_set": ["sn0", "sn2", "sn1", "sw0", "sw2", "sw1"],
    "mse_bound": [sn0_mse_bound, sn2_mse_bound, sn1_mse_bound, sw0_mse_bound, sw2_mse_bound, sw1_mse_bound]})
mse_bounds.set_index("data_set", inplace=True)
mse_bounds

## Analysis of results from the narrow datasets

### One period, zero noise

In [ ]:
sns.displot(data=results_sn0, x="mse", col="Lopstr", log_scale=True)

In [ ]:
sns.displot(data=results_sn0, x="complexity", col="Lopstr")

With no noise, the function $\sin x$ is recovered every time, apart from fuzz.

In [ ]:
sns.displot(data=results_sn0, x="complexity_defuzz", col="Lopstr")

### One period, low noise

In [ ]:
sns.displot(data=results_sn2, x="mse", col="Lopstr", log_scale=True)

In [ ]:
fig = sns.displot(data=results_sn2, x="complexity_defuzz", y="mse", col="Lopstr", log_scale=[False, True])
# Add a horizontal line for the MSE bound in each subplot
for ax in fig.axes.flat:
    ax.axhline(y=sn2_mse_bound, color="red", linestyle="--", label="MSE bound")
fig

In [ ]:
results_sn2

In [ ]:
au.count_by_threshold(results_sn2, 0.9*sn2_mse_bound, groupby="Lopstr")

In [ ]:
sns.displot(data=results_sn2, x="complexity", col="Lopstr")

In [ ]:
sns.displot(data=results_sn2, x="complexity_defuzz", col="Lopstr")

In [ ]:
exprs_sn2 = results_sn2.sympy

In [ ]:
exprs_sn2

In [ ]:
exprs_sn2.apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-2))

Exact recovery isn't too bad.
These are essentially correct, but it's hard to see.
There's a lot of cruft, and things like $-\cos(x + \pi/2)$, $\sin(x + \varepsilon)$, etc. instead of $\sin x$. 

In [ ]:
results_sn2

In [ ]:
au.count_by_threshold(results_sn2, 0.9*sn2_mse_bound, groupby="Lopstr")

In [ ]:
sns.displot(data=results_sn2, x="complexity", col="Lopstr")

In [ ]:
sns.displot(data=results_sn2, x="complexity_defuzz", col="Lopstr")

In [ ]:
exprs_sn2 = results_sn2.sympy

In [ ]:
exprs_sn2

In [ ]:
exprs_sn2.apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-2))

Exact recovery isn't too bad.
These are essentially correct, but it's hard to see.
There's a lot of cruft, and things like $-\cos(x + \pi/2)$, $\sin(x + \varepsilon)$, etc. instead of $\sin x$. 

### One period, high noise

In [ ]:
sns.displot(data=results_sn1, x="mse", col="Lopstr", log_scale=True)

In [ ]:
fig = sns.displot(data=results_sn1, x="complexity_defuzz", y="mse", col="Lopstr", log_scale=[False, True])
for ax in fig.axes.flat:
    ax.axhline(y=sn1_mse_bound, color="red", linestyle="--", label="MSE bound")
fig

In [ ]:
results_sn1

In [ ]:
au.count_by_threshold(results_sn1, 0.9*sn1_mse_bound, groupby="Lopstr")

In [ ]:
sns.displot(data=results_sn1, x="complexity", col="Lopstr")

In [ ]:
sns.displot(data=results_sn1, x="complexity_defuzz", col="Lopstr")

In [ ]:
exprs_sn1 = results_sn1.sympy

In [ ]:
exprs_sn1

In [ ]:
exprs_sn1.apply(lambda e: au.replace_near_integer(e, tolerance=1.0e-2))

Many of these aren't bad, but the cruft is significant if $\lambda$ is too small.

So for low noise, increasing $\lambda_{{\mathrm{op}}}$ definitely lowers the complexity without increasing the MSE past the known correct value.

So the story here is that if there's noise, the complexity measure has to be of approximately the same order of magnitude as the noise, otherwise, it will add a lot of cruft trying to fit the noise.
The effect is very noticeable.
Which means that you have to estimate the noise before running symbolic regression.